
#A Study of Consistency Models





## 2D Toy Dataset : make_moons

###Setup, Hyperparameters, and Dataset

In [3]:
!pip install --upgrade POT jax jaxlib
!pip install POT
import numpy as np
import torch
import torch.nn as nn
import os
import ot
from scipy.spatial.distance import cdist
import shutil
from datetime import datetime
from google.colab import drive
import pandas as pd
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy
from IPython.display import clear_output

In [4]:
DATA_SEED = 42
TEACHER_SEED = 333
SEED = 1234567890 # for CMs training
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

N_SAMPLES       = 8_000
NOISE_LEVEL     = 0.08      # sklearn noise parameter for make_moons

#diffusion model param
SIGMA_DATA      = 1       # σ_data
T_MIN           = 0.002     # ε
T_MAX           = 10.0      # T: maximum noise level
RHO             = 7.0       # ρ: exponent for the Karras time schedule
LR_DIFFUSION        = 1e-3
EPOCHS_DIFFUSION    = 200

#consistency models param
N_DISC          = 80       # N: number of discrete time steps in schedule
EMA_DECAY       = 0.95    # μ: EMA decay for the target (θ̄) network
LR_CD               = 5*1e-3
LR_CT               = 1e-3
EPOCHS_CD           = 500
EPOCHS_CT           = 4000

BATCH_SIZE          = 512

DATA_DIM        = 2
HIDDEN_DIM      = 256
TIME_EMB_DIM    = 128
N_RESIDUAL      = 4         # number of residual blocks in the backbone

#to get the dataset
def get_moons_dataset(n_samples: int = N_SAMPLES, noise: float = NOISE_LEVEL,seed = DATA_SEED) -> torch.Tensor:
    X, _ = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
    X = X.astype(np.float32)
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)   # zero-mean, unit-std
    return torch.tensor(X, dtype=torch.float32, device=DEVICE)

data = get_moons_dataset()
print(f"Dataset shape: {data.shape}")



Device: cuda
Dataset shape: torch.Size([8000, 2])


###Neural Network Backbone




In [6]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int = TIME_EMB_DIM):
        super().__init__()
        assert dim % 2 == 0
        half = dim // 2
        freqs = torch.exp(-np.log(10_000.0) * torch.arange(half, dtype=torch.float32) / (half - 1))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        t   = t.reshape(-1, 1)
        arg = t * self.freqs.unsqueeze(0)
        emb = torch.cat([arg.sin(), arg.cos()], dim=-1)
        return self.mlp(emb)

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim: int = HIDDEN_DIM, time_emb_dim: int = TIME_EMB_DIM):
        super().__init__()
        self.norm      = nn.LayerNorm(hidden_dim)
        self.linear    = nn.Linear(hidden_dim, hidden_dim)
        self.act       = nn.SiLU()
        self.time_proj = nn.Linear(time_emb_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        h = self.linear(h) + self.time_proj(t_emb)
        h = self.act(h)
        return x + h

class ScoreNetwork(nn.Module):
    def __init__(self, data_dim: int = DATA_DIM, hidden_dim: int = HIDDEN_DIM,
                 time_emb_dim: int = TIME_EMB_DIM, n_hidden: int = N_RESIDUAL):
        super().__init__()
        self.time_emb   = SinusoidalTimeEmbedding(time_emb_dim)
        self.input_proj = nn.Sequential(nn.Linear(data_dim, hidden_dim), nn.SiLU())
        self.blocks     = nn.ModuleList([ResidualBlock(hidden_dim, time_emb_dim) for _ in range(n_hidden)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_emb(t)
        h     = self.input_proj(x)
        for block in self.blocks:
            h = block(h, t_emb)
        return self.output_proj(h)



###EDM Preconditioning & Diffusion Teacher




In [8]:
def edm_precond(t: torch.Tensor, sigma_data: float = SIGMA_DATA) -> dict:
    sd2     = sigma_data ** 2
    c_skip  = sd2 / (t**2 + sd2)
    c_out   = t * sigma_data / (t**2 + sd2).sqrt()
    c_in    = 1.0  / (t**2 + sd2).sqrt()
    c_noise = 0.25 * torch.log(t.clamp(min=1e-8))
    return dict(c_skip=c_skip, c_out=c_out, c_in=c_in, c_noise=c_noise)

class EDMDenoiser(nn.Module):
    def __init__(self, backbone: ScoreNetwork, sigma_data: float = SIGMA_DATA):
        super().__init__()
        self.backbone   = backbone
        self.sigma_data = sigma_data

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pc  = edm_precond(t, self.sigma_data)
        cs  = pc["c_skip"].unsqueeze(1)
        co  = pc["c_out"].unsqueeze(1)
        ci  = pc["c_in"].unsqueeze(1)
        cn  = pc["c_noise"]

        F   = self.backbone(ci * x_t, cn)
        return cs * x_t + co * F

    def pf_ode_derivative(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t = t.reshape(-1)
        x0_hat = self.forward(x_t, t)
        return (x_t - x0_hat) / t.unsqueeze(1)

@torch.no_grad()
def sample_diffusion(model, n, n_steps=100, t_min=T_MIN, t_max=T_MAX, rho=RHO):
    ts = build_time_schedule(n_steps + 1, t_min, t_max, rho).flip(0)
    x = torch.randn(n, 2, device=DEVICE) * t_max
    for i in range(n_steps):
        t_cur = ts[i].expand(n)
        t_next = ts[i+1].expand(n)
        dt = (t_next - t_cur).unsqueeze(1)
        d1 = model.pf_ode_derivative(x, t_cur)
        x_euler = x + dt * d1
        d2 = model.pf_ode_derivative(x_euler, t_next)
        x = x + dt * 0.5 * (d1 + d2)
    return x

###Consistency Model Preconditioning & Helpers

(defines the boundary-enforcing wrapper for Consistency Models, the time schedule, and the EMA update function)



In [9]:
def cm_precond(t: torch.Tensor, sigma_data: float = SIGMA_DATA) -> dict:
    sd2 = sigma_data ** 2

    c_skip = sd2 / (t**2 + sd2)
    c_out  = t * sigma_data / torch.sqrt(t**2 + sd2)
    c_in   = 1.0 / torch.sqrt(t**2 + sd2)
    c_noise = 0.25 * torch.log(t.clamp(min=1e-8))

    return dict(
        c_skip=c_skip,
        c_out=c_out,
        c_in=c_in,
        c_noise=c_noise
    )

class ConsistencyModel(nn.Module):
    def __init__(self, backbone: ScoreNetwork, sigma_data: float = SIGMA_DATA):
        super().__init__()
        self.backbone = backbone
        self.sigma_data = sigma_data

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pc = cm_precond(t, self.sigma_data)

        cs = pc["c_skip"].unsqueeze(1)
        co = pc["c_out"].unsqueeze(1)
        ci = pc["c_in"].unsqueeze(1)
        cn = pc["c_noise"]

        F = self.backbone(ci * x, cn)

        return cs * x + co * F

def build_time_schedule(N: int, t_min: float = T_MIN, t_max: float = T_MAX, rho: float = RHO) -> torch.Tensor:
    if N == 1:
        return torch.tensor([t_max], device=DEVICE)
    i = torch.arange(N, device=DEVICE)
    ts = (t_min**(1.0/rho) + (i / (N - 1)) * (t_max**(1.0/rho) - t_min**(1.0/rho))) ** rho
    return ts.to(DEVICE)

def ema_update(online: nn.Module, target: nn.Module, decay: float) -> None:
    with torch.no_grad():
        for p_tgt, p_on in zip(target.parameters(), online.parameters()):
            p_tgt.mul_(decay).add_(p_on.detach(), alpha=1.0 - decay)

### Training Loops
(one for the Teacher, one for CD, and one for CT)



In [10]:

def train_diffusion(data: torch.Tensor, epochs: int = EPOCHS_DIFFUSION, batch_size: int = BATCH_SIZE,
                    lr: float = LR_DIFFUSION, sigma_data: float = SIGMA_DATA, t_min: float = T_MIN, t_max: float = T_MAX):
    backbone  = ScoreNetwork().to(DEVICE)
    model     = EDMDenoiser(backbone, sigma_data=sigma_data).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loader  = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True, drop_last=True)
    losses  = []

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for (x0,) in loader:
            B = x0.shape[0]
            u = torch.rand(B, device=DEVICE)
            t = (t_min ** (1 / RHO) + u * (t_max ** (1 / RHO) - t_min ** (1 / RHO))) ** RHO
            eps = torch.randn_like(x0)
            x_t = x0 + t.unsqueeze(1) * eps
            lam = (t**2 + sigma_data**2) / (t * sigma_data)**2

            x0_hat = model(x_t, t)
            loss   = (lam.unsqueeze(1) * (x0_hat - x0)**2).mean()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        if (epoch + 1) % 50 == 0:
            print(f"  [Diffusion] Epoch {epoch+1:4d}/{epochs}  loss = {avg:.5f}")
    return model, losses

def song_N_schedule(k: int, K: int, s0: int = 2, s: int = N_DISC) -> int:
    inner = k * (s**2 - s0**2) / K + s0**2
    return min(s, int(np.ceil(np.sqrt(inner) - 0.25)) + 1)


def song_mu_schedule(N_curr: int, mu0: float = EMA_DECAY, s0: int = 2) -> float:
    return float(np.exp(s0 * np.log(mu0) / N_curr))



def train_consistency_distillation(teacher: EDMDenoiser, data: torch.Tensor, N: int = N_DISC, epochs: int = EPOCHS_CD,
                                   batch_size: int = BATCH_SIZE, lr: float = LR_CD, mu: float = EMA_DECAY, t_min: float = T_MIN,
                                   t_max: float = T_MAX, sigma_data: float = SIGMA_DATA):
    online_bb = ScoreNetwork().to(DEVICE)
    ema_bb = ScoreNetwork().to(DEVICE)
    ema_bb.load_state_dict(online_bb.state_dict())
    online_cm = ConsistencyModel(online_bb, sigma_data=sigma_data).to(DEVICE)
    ema_cm    = ConsistencyModel(ema_bb, sigma_data=sigma_data).to(DEVICE)
    for p in ema_cm.parameters(): p.requires_grad_(False)

    optimizer = optim.Adam(online_cm.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loader  = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True, drop_last=True)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    losses = []

    # K = total number of steps (for the schedule of Song et al.)
    K = epochs * len(loader)
    global_step = 0

    for epoch in range(epochs):
        epoch_loss = 0.0
        for (x0,) in loader:
            N_curr = song_N_schedule(global_step, K, s0=2, s=N)
            ts = build_time_schedule(N_curr, t_min=t_min, t_max=t_max)

            B = x0.shape[0]
            n_idx = torch.randint(0, N_curr - 1, (B,), device=DEVICE)
            t_n   = ts[n_idx]
            t_n1  = ts[n_idx + 1]

            eps   = torch.randn_like(x0)
            x_n1  = x0 + t_n1.unsqueeze(1) * eps
            with torch.no_grad():
                d     = teacher.pf_ode_derivative(x_n1, t_n1)
                dt    = (t_n - t_n1).unsqueeze(1)
                x_n_hat = x_n1 + dt * d

            pred_n1 = online_cm(x_n1,   t_n1)
            with torch.no_grad():
                tgt_n   = ema_cm(x_n_hat, t_n)

            # λ(t) = 1 like in the paper's experiments
            loss = ((pred_n1 - tgt_n)**2).mean()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online_cm.parameters(), max_norm=1.0)
            optimizer.step()
            ema_update(online_bb, ema_bb, mu)  # constant μ for CD
            epoch_loss += loss.item()
            global_step += 1

        scheduler.step()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        if (epoch + 1) % 50 == 0:
            N_log = song_N_schedule(global_step - 1, K, s0=2, s=N)
            print(f"  [CD] Epoch {epoch+1:4d}/{epochs}  loss = {avg:.6f}  N_curr={N_log}")
    return online_cm, losses

def train_consistency_training(data: torch.Tensor, N: int = N_DISC, epochs: int = EPOCHS_CT, batch_size: int = BATCH_SIZE,
                               lr: float = LR_CT, mu: float = EMA_DECAY, t_min: float = T_MIN, t_max: float = T_MAX, sigma_data: float = SIGMA_DATA):
    online_bb = ScoreNetwork().to(DEVICE)
    ema_bb = ScoreNetwork().to(DEVICE)
    ema_bb.load_state_dict(online_bb.state_dict())
    online_cm = ConsistencyModel(online_bb, sigma_data=sigma_data).to(DEVICE)
    ema_cm    = ConsistencyModel(ema_bb, sigma_data=sigma_data).to(DEVICE)
    for p in ema_cm.parameters(): p.requires_grad_(False)

    optimizer = optim.Adam(online_cm.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loader  = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True, drop_last=True)
    losses  = []

    # K = total number of steps (for the scheduling)
    K = epochs * len(loader)
    global_step = 0

    for epoch in range(epochs):
        epoch_loss = 0.0
        for (x0,) in loader:
            # schedule for N
            N_curr = song_N_schedule(global_step, K, s0=2, s=N)
            ts = build_time_schedule(N_curr, t_min=t_min, t_max=t_max)
            mu_curr = song_mu_schedule(N_curr, mu0=mu, s0=2)

            B = x0.shape[0]
            n_idx = torch.randint(0, N_curr - 1, (B,), device=DEVICE)
            t_n   = ts[n_idx]
            t_n1  = ts[n_idx + 1]

            z      = torch.randn_like(x0)
            x_tn   = x0 + t_n.unsqueeze(1)  * z
            x_tn1  = x0 + t_n1.unsqueeze(1) * z

            pred = online_cm(x_tn1, t_n1)
            with torch.no_grad():
                tgt  = ema_cm(x_tn, t_n)

            # λ(t) = 1 like in the paper's experiments
            loss = ((pred - tgt)**2).mean()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online_cm.parameters(), max_norm=1.0)
            optimizer.step()
            ema_update(online_bb, ema_bb, mu_curr)  # dynamic μ for CT
            epoch_loss += loss.item()
            global_step += 1

        scheduler.step()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        if (epoch + 1) % 50 == 0:
            N_log  = song_N_schedule(global_step - 1, K, s0=2, s=N)
            mu_log = song_mu_schedule(N_log, mu0=mu, s0=2)
            print(f"  [CT] Epoch {epoch+1:4d}/{epochs}  loss = {avg:.6f}  N_curr={N_log}  μ={mu_log:.4f}")
    return online_cm, losses


###Sampling & Visualization Functions




In [11]:
def compute_w1(real_pts: np.ndarray, gen_pts: np.ndarray) -> float:
    M = cdist(real_pts, gen_pts, metric='euclidean')
    a, b = np.ones((len(real_pts),)) / len(real_pts), np.ones((len(gen_pts),)) / len(gen_pts)
    return ot.emd2(a, b, M)

@torch.no_grad()
def sample_consistency_1step(model: ConsistencyModel, n: int, t_max: float = T_MAX) -> torch.Tensor:
    model.eval()
    x_T = torch.randn(n, 2, device=DEVICE) * t_max
    t   = torch.full((n,), t_max, device=DEVICE)
    return model(x_T, t)

@torch.no_grad()
def sample_consistency_multistep(model, n, n_steps,
                                 t_min=T_MIN,
                                 t_max=T_MAX,
                                 rho=RHO):
    ts = build_time_schedule(n_steps + 1, t_min, t_max, rho).flip(0)
    x = torch.randn(n, 2, device=DEVICE) * t_max

    for i in range(n_steps):
        t_cur = ts[i].expand(n)
        x0_hat = model(x, t_cur)
        if i < n_steps - 1:
            t_next = ts[i + 1]
            z = torch.randn_like(x0_hat)
            x = x0_hat + t_next * z
        else:
            x = x0_hat

    return x

def plot_training_curves(losses_diff: list, losses_cd: list, losses_ct: list) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.patch.set_facecolor("#0d0d1a")
    panels = [
        (losses_diff, "Diffusion Teacher (EDM)", "#64b5f6"),
        (losses_cd,   "Consistency Distillation (CD)", "#81c784"),
        (losses_ct,   "Consistency Training (CT)",     "#ffb74d"),
    ]
    for ax, (losses, title, color) in zip(axes, panels):
        ax.set_facecolor("#12122b")
        ax.plot(losses, color=color, linewidth=1.8, alpha=0.9)
        window = max(1, len(losses) // 20)
        if len(losses) >= window:
            kernel = np.ones(window) / window
            smooth = np.convolve(losses, kernel, mode="valid")
            ax.plot(range(window-1, len(losses)), smooth, color="white", linewidth=1.0, alpha=0.5, linestyle="--", label="smoothed")
        ax.set_title(title, color="white", fontsize=11, pad=8)
        ax.set_xlabel("Epoch", color="#aaaacc", fontsize=9)
        ax.set_ylabel("Loss",  color="#aaaacc", fontsize=9)
        ax.tick_params(colors="#666688", labelsize=8)
        for spine in ax.spines.values(): spine.set_edgecolor("#22224a")
        ax.legend(fontsize=8, labelcolor="white", facecolor="#0d0d1a", edgecolor="none")
    fig.suptitle("Training Loss Curves", color="white", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

def plot_samples(real_data: torch.Tensor, teacher: EDMDenoiser, cd_model: ConsistencyModel, ct_model: ConsistencyModel, n_samples: int = 2000) -> np.ndarray:
    print("\nSampling for visualization")
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    real_np = real_data[:n_samples].cpu().numpy()

    with torch.no_grad():
        t_samples = sample_diffusion(teacher, n_samples, n_steps=100).cpu().numpy()
        cd_1  = sample_consistency_1step(cd_model, n_samples).cpu().numpy()
        cd_2  = sample_consistency_multistep(cd_model, n_samples, n_steps=2).cpu().numpy()
        cd_3  = sample_consistency_multistep(cd_model, n_samples, n_steps=3).cpu().numpy()
        cd_5  = sample_consistency_multistep(cd_model, n_samples, n_steps=5).cpu().numpy()
        cd_10 = sample_consistency_multistep(cd_model, n_samples, n_steps=10).cpu().numpy()
        ct_1  = sample_consistency_1step(ct_model, n_samples).cpu().numpy()
        ct_2  = sample_consistency_multistep(ct_model, n_samples, n_steps=2).cpu().numpy()
        ct_3  = sample_consistency_multistep(ct_model, n_samples, n_steps=3).cpu().numpy()
        ct_5  = sample_consistency_multistep(ct_model, n_samples, n_steps=5).cpu().numpy()
        ct_10 = sample_consistency_multistep(ct_model, n_samples, n_steps=10).cpu().numpy()

    print("W1 computation")
    w_t    = compute_w1(real_np, t_samples)
    w_cd1  = compute_w1(real_np, cd_1)
    w_cd2  = compute_w1(real_np, cd_2)
    w_cd3  = compute_w1(real_np, cd_3)
    w_cd5  = compute_w1(real_np, cd_5)
    w_cd10 = compute_w1(real_np, cd_10)
    w_ct1  = compute_w1(real_np, ct_1)
    w_ct2  = compute_w1(real_np, ct_2)
    w_ct3  = compute_w1(real_np, ct_3)
    w_ct5  = compute_w1(real_np, ct_5)
    w_ct10 = compute_w1(real_np, ct_10)

    vector_w1 = np.array([w_t, w_cd1, w_cd2, w_cd3, w_cd5, w_cd10, w_ct1, w_ct2, w_ct3, w_ct5, w_ct10])

    panels = [
        (real_np, "Real Data\n(make_moons)", "#4fc3f7"),
        (cd_1,    f"CD 1-step\n$W_1$={w_cd1:.4f}", "#ffb74d"),
        (cd_2,    f"CD 2-step\n$W_1$={w_cd2:.4f}", "#ffa726"),
        (cd_3,    f"CD 3-step\n$W_1$={w_cd3:.4f}", "#ff9800"),
        (cd_5,    f"CD 5-step\n$W_1$={w_cd5:.4f}", "#fb8c00"),
        (cd_10,   f"CD 10-step\n$W_1$={w_cd10:.4f}", "#f57c00"),

        (t_samples, f"Teacher (100 steps)\n$W_1$={w_t:.4f}", "#81c784"),
        (ct_1,    f"CT 1-step\n$W_1$={w_ct1:.4f}", "#f48fb1"),
        (ct_2,    f"CT 2-step\n$W_1$={w_ct2:.4f}", "#f06292"),
        (ct_3,    f"CT 3-step\n$W_1$={w_ct3:.4f}", "#e91e63"),
        (ct_5,    f"CT 5-step\n$W_1$={w_ct5:.4f}", "#d81b60"),
        (ct_10,   f"CT 10-step\n$W_1$={w_ct10:.4f}", "#c2185b"),
    ]

    fig = plt.figure(figsize=(22, 8))
    fig.patch.set_facecolor("#0d0d1a")
    gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.35, wspace=0.25)
    lim = 2.8
    for idx, (pts, title, color) in enumerate(panels):
        row, col = divmod(idx, 6)
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor("#12122b")
        ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.45, color=color, rasterized=True, linewidths=0)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_title(title, color="white", fontsize=11, pad=8)
        ax.tick_params(colors="#666688", labelsize=8)
        ax.set_aspect("equal")
        for spine in ax.spines.values(): spine.set_edgecolor("#22224a")

    fig.suptitle("Evolution of Generation Quality over Multiple Steps (with W1 Distance)", color="white", fontsize=14, fontweight="bold", y=1.03)
    plt.savefig("consistency_models_moons1.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    return vector_w1

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.9.1, so it will not be used.
  warnings.warn(


###Execute Pipeline




In [12]:
print("=== Phase 1: Training Diffusion Teacher (EDM) ===")
teacher, losses_diff = train_diffusion(data)

=== Phase 1: Training Diffusion Teacher (EDM) ===
  [Diffusion] Epoch   50/200  loss = 2.06510


KeyboardInterrupt: 

In [ ]:
print("\n=== Phase 2: Training Consistency Distillation (CD) ===")
cd_model, losses_cd = train_consistency_distillation(teacher, data)

print("\n=== Phase 3: Training Consistency Training (CT) ===")
ct_model, losses_ct = train_consistency_training(data)

print("\n=== Phase 4: Visualizing Results ===")
plot_training_curves(losses_diff, losses_cd, losses_ct)
w1_vector = plot_samples(data, teacher, cd_model, ct_model, n_samples=2000)

To secure the results :  


In [ ]:
DRIVE_FOLDER = '/content[to complete]'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%Mm%Ss")
params_str = (
    f"Seed-{SEED}_"
    f"Data-{N_SAMPLES}_Nz-{NOISE_LEVEL}_"
    f"Sig-{SIGMA_DATA}_Tmin-{T_MIN}_Tmax-{T_MAX}_Rho-{RHO}_"
    f"Ndisc-{N_DISC}_EMA-{EMA_DECAY}_"
    f"BS-{BATCH_SIZE}_"
    f"LRd-{LR_DIFFUSION}_LRcd-{LR_CD}_LRct-{LR_CT}_"
    f"Epd-{EPOCHS_DIFFUSION}_Epcd-{EPOCHS_CD}_Epct-{EPOCHS_CT}_"
    f"Dim-{HIDDEN_DIM}_Tdim-{TIME_EMB_DIM}_Res-{N_RESIDUAL}"
)

nom_fichier_courbes = f"courbes_{params_str}_{timestamp}.png"
nom_fichier_moons   = f"samples_{params_str}_{timestamp}.png"

chemin_drive_courbes = os.path.join(DRIVE_FOLDER, nom_fichier_courbes)
chemin_drive_moons   = os.path.join(DRIVE_FOLDER, nom_fichier_moons)

if os.path.exists("training_curves.png"):
    shutil.copy("training_curves.png", chemin_drive_courbes)
    print(f"curves saved : {nom_fichier_courbes}")

if os.path.exists("consistency_models_moons1.png"):
    shutil.copy("consistency_models_moons1.png", chemin_drive_moons)
    print(f"samples saved : {nom_fichier_moons}")

#màj csv
master_csv_path = os.path.join(DRIVE_FOLDER, "MASTER_METRICS_Global.csv")

nouvelle_run = {
    'Seed': SEED, 'Data': N_SAMPLES, 'Nz': NOISE_LEVEL,
    'Sig': SIGMA_DATA, 'Tmin': T_MIN, 'Tmax': T_MAX, 'Rho': RHO,
    'N_DISC': N_DISC, 'EMA_DECAY': EMA_DECAY, 'BS': BATCH_SIZE,
    'LRd': LR_DIFFUSION, 'LR_CD': LR_CD, 'LR_CT': LR_CT,
    'Epd': EPOCHS_DIFFUSION, 'EPOCHS_CD': EPOCHS_CD, 'EPOCHS_CT': EPOCHS_CT,
    'Dim': HIDDEN_DIM, 'Tdim': TIME_EMB_DIM, 'Res': N_RESIDUAL,
    'Timestamp': timestamp
}

en_tete = "Teacher_100,CD_1,CD_2,CD_3,CD_5,CD_10,CT_1,CT_2,CT_3,CT_5,CT_10"
for k, val in zip(en_tete.split(','), w1_vector):
    nouvelle_run[f"W1_{k}"] = val

df_new = pd.DataFrame([nouvelle_run])

if os.path.exists(master_csv_path):
    df_old = pd.read_csv(master_csv_path)
    df_combined = pd.concat([df_old, df_new], ignore_index=True)
    df_combined.to_csv(master_csv_path, index=False)
    print(f"MASTER_METRICS_Global.csv mis à jour")



Additional visualization functions  :

In [ ]:
def plot_and_eval_teacher(real_data, teacher, n_samples=2000):
    print("\nTeacher's evaluation (1, 2, 3, 5, 10, 50, 100 steps)...")
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    real_np = real_data[:n_samples].cpu().numpy()

    steps = [1, 2, 3, 5, 10, 50, 100]
    results_w1 = {}
    panels = [(real_np, "Real Data\n(make_moons)", "#4fc3f7")]

    with torch.no_grad():
        for stp in steps:
            gen = sample_diffusion(teacher, n_samples, n_steps=stp).cpu().numpy()
            w1 = compute_w1(real_np, gen)
            results_w1[f"W1_Teacher_{stp}"] = w1
            panels.append((gen, f"Teacher {stp} steps\n$W_1$={w1:.4f}", "#81c784"))

    fig = plt.figure(figsize=(16, 8))
    fig.patch.set_facecolor("#0d0d1a")
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.25)
    lim = 2.8
    for idx, (pts, title, color) in enumerate(panels):
        row, col = divmod(idx, 4)
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor("#12122b")
        ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.45, color=color, rasterized=True, linewidths=0)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_title(title, color="white", fontsize=11, pad=8)
        ax.tick_params(colors="#666688", labelsize=8)
        ax.set_aspect("equal")
        for spine in ax.spines.values(): spine.set_edgecolor("#22224a")

    fig.suptitle("Teacher (Diffusion) Generation Quality over Multiple Steps", color="white", fontsize=14, fontweight="bold", y=0.96)
    plt.savefig("teacher_moons.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    return results_w1


def plot_and_eval_cm(real_data, cd_model, ct_model, n_samples=2000):
    print("CMs evaluation (1, 2, 3, 5, 10 steps)...")
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    real_np = real_data[:n_samples].cpu().numpy()

    steps = [1, 2, 3, 5, 10]
    results_w1 = {}

    # CD
    panels = [(real_np, "Real Data (CD)\n(make_moons)", "#4fc3f7")]
    colors_cd = ["#ffb74d", "#ffa726", "#ff9800", "#fb8c00", "#f57c00"]
    with torch.no_grad():
        for i, stp in enumerate(steps):
            gen = sample_consistency_1step(cd_model, n_samples).cpu().numpy() if stp == 1 else sample_consistency_multistep(cd_model, n_samples, n_steps=stp).cpu().numpy()
            w1 = compute_w1(real_np, gen)
            results_w1[f"W1_CD_{stp}"] = w1
            panels.append((gen, f"CD {stp}-step\n$W_1$={w1:.4f}", colors_cd[i]))

    # CT
    panels.append((real_np, "Real Data (CT)\n(make_moons)", "#4fc3f7"))
    colors_ct = ["#f48fb1", "#f06292", "#e91e63", "#d81b60", "#c2185b"]
    with torch.no_grad():
        for i, stp in enumerate(steps):
            gen = sample_consistency_1step(ct_model, n_samples).cpu().numpy() if stp == 1 else sample_consistency_multistep(ct_model, n_samples, n_steps=stp).cpu().numpy()
            w1 = compute_w1(real_np, gen)
            results_w1[f"W1_CT_{stp}"] = w1
            panels.append((gen, f"CT {stp}-step\n$W_1$={w1:.4f}", colors_ct[i]))

    fig = plt.figure(figsize=(22, 8))
    fig.patch.set_facecolor("#0d0d1a")
    gs  = gridspec.GridSpec(2, 6, figure=fig, hspace=0.35, wspace=0.25)
    lim = 2.8
    for idx, (pts, title, color) in enumerate(panels):
        row, col = divmod(idx, 6)
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor("#12122b")
        ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.45, color=color, rasterized=True, linewidths=0)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_title(title, color="white", fontsize=11, pad=8)
        ax.tick_params(colors="#666688", labelsize=8)
        ax.set_aspect("equal")
        for spine in ax.spines.values(): spine.set_edgecolor("#22224a")

    fig.suptitle("Consistency Models Generation Quality over Multiple Steps", color="white", fontsize=14, fontweight="bold", y=1.03)
    plt.savefig("consistency_models_moons.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    return results_w1